# Weather & Climate Trend Analysis

Analyze temperature and precipitation trends across years and seasons for Oregon Willamette Valley agricultural fields.

**Questions to Answer:**
- Are summers getting hotter?
- Is winter rain decreasing?
- What are the overall climate trends?
- How is Growing Degree Days (GDD) changing?

In [2]:
# Setup and Data Loading
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import numpy as np

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

# Load monthly summary data
monthly = pd.read_csv('data/weather_monthly_summary.csv')
print(f"Loaded {len(monthly):,} monthly records")
print(f"Fields: {monthly['field_id'].nunique()}")
print(f"Years: {monthly['year'].min()} - {monthly['year'].max()}")
print(f"\nColumns: {list(monthly.columns)}")

FileNotFoundError: [Errno 2] No such file or directory: 'data/weather_monthly_summary.csv'

In [ ]:
# Display basic info
print("Data sample:")
monthly.head(10)

## 2. Annual Temperature Trends

Analyze how average annual temperature is changing over time.

In [ ]:
# Calculate annual temperature averages across all fields
annual_temp = monthly.groupby('year').agg({
    'temp_mean_c': 'mean',
    'temp_max_mean_c': 'mean',
    'temp_min_mean_c': 'mean'
}).reset_index()

print("Annual Temperature Summary:")
annual_temp

In [ ]:
# Plot annual temperature trend with regression
fig, ax = plt.subplots(figsize=(12, 6))

# Plot data
ax.plot(annual_temp['year'], annual_temp['temp_mean_c'], 'o-', 
        label='Mean Temperature', linewidth=2, markersize=6)
ax.fill_between(annual_temp['year'], 
                annual_temp['temp_min_mean_c'], 
                annual_temp['temp_max_mean_c'], 
                alpha=0.3, label='Min-Max Range')

# Add regression line
slope, intercept, r_value, p_value, std_err = stats.linregress(
    annual_temp['year'], annual_temp['temp_mean_c']
)
ax.plot(annual_temp['year'], intercept + slope * annual_temp['year'], 
        'r--', linewidth=2, label=f'Trend: {slope:.3f}°C/year')

ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Temperature (°C)', fontsize=12)
ax.set_title('Annual Average Temperature Trend (2000-2025)', fontsize=14)
ax.legend()
ax.set_ylim(10, 16)

# Add statistical info
ax.text(0.02, 0.98, f'R² = {r_value**2:.3f}\np-value = {p_value:.4f}', 
        transform=ax.transAxes, fontsize=10, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig('notebooks/output/annual_temp_trend.png', dpi=150)
plt.show()

print(f"\nTrend Analysis:")
print(f"  Rate of change: {slope:.4f}°C per year")
print(f"  Total change over 26 years: {slope * 26:.2f}°C")
print(f"  Statistical significance: p = {p_value:.4f} ({'significant' if p_value < 0.05 else 'not significant'})")

## 3. Seasonal Temperature Analysis

Compare summer and winter temperatures across years to see if seasons are changing differently.

In [ ]:
# Seasonal temperature analysis
seasonal_temp = monthly.groupby(['year', 'season']).agg({
    'temp_mean_c': 'mean'
}).reset_index()

# Pivot for easier plotting
seasonal_pivot = seasonal_temp.pivot(index='year', columns='season', values='temp_mean_c')
seasonal_pivot = seasonal_pivot[['winter', 'spring', 'summer', 'fall']]  # Order seasons

print("Seasonal Temperatures by Year:")
seasonal_pivot.round(2)

In [ ]:
# Plot seasonal temperatures
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
seasons = ['winter', 'spring', 'summer', 'fall']

for idx, season in enumerate(seasons):
    ax = axes[idx // 2, idx % 2]
    
    season_data = seasonal_temp[seasonal_temp['season'] == season]
    
    ax.plot(season_data['year'], season_data['temp_mean_c'], 'o-', 
            linewidth=2, markersize=5)
    
    # Add trend line
    slope, intercept, r_value, p_value, std_err = stats.linregress(
        season_data['year'], season_data['temp_mean_c']
    )
    ax.plot(season_data['year'], intercept + slope * season_data['year'], 
            'r--', linewidth=2, alpha=0.7)
    
    ax.set_title(f'{season.capitalize()} Temperature', fontsize=12)
    ax.set_xlabel('Year')
    ax.set_ylabel('Temperature (°C)')
    ax.text(0.02, 0.98, f'Trend: {slope:.4f}°C/yr\np = {p_value:.3f}', 
            transform=ax.transAxes, fontsize=9, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle('Seasonal Temperature Trends (2000-2025)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('notebooks/output/seasonal_temp_trends.png', dpi=150)
plt.show()

# Summary of seasonal trends
print("\nSeasonal Temperature Trend Summary:")
print("-" * 60)
for season in seasons:
    season_data = seasonal_temp[seasonal_temp['season'] == season]
    slope, _, r_value, p_value, _ = stats.linregress(
        season_data['year'], season_data['temp_mean_c']
    )
    sig = "*" if p_value < 0.05 else ""
    print(f"  {season.capitalize():8s}: {slope:+.4f}°C/year (p={p_value:.3f}) {sig}")

## 4. Precipitation Trends

Analyze annual and seasonal precipitation patterns.

In [ ]:
# Annual precipitation totals
annual_precip = monthly.groupby('year').agg({
    'precip_total_mm': 'sum'
}).reset_index()

print("Annual Precipitation:")
annual_precip

In [ ]:
# Plot annual precipitation
fig, ax = plt.subplots(figsize=(12, 6))

ax.bar(annual_precip['year'], annual_precip['precip_total_mm'], 
       color='steelblue', alpha=0.7)

# Add trend line
slope, intercept, r_value, p_value, std_err = stats.linregress(
    annual_precip['year'], annual_precip['precip_total_mm']
)
ax.plot(annual_precip['year'], intercept + slope * annual_precip['year'], 
        'r-', linewidth=2, label=f'Trend: {slope:.1f} mm/year')

ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Annual Precipitation (mm)', fontsize=12)
ax.set_title('Annual Precipitation Trend (2000-2025)', fontsize=14)
ax.legend()

ax.text(0.02, 0.98, f'R² = {r_value**2:.3f}\np-value = {p_value:.4f}', 
        transform=ax.transAxes, fontsize=10, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig('notebooks/output/annual_precip_trend.png', dpi=150)
plt.show()

print(f"\nPrecipitation Trend Analysis:")
print(f"  Rate of change: {slope:.2f} mm per year")
print(f"  Total change over 26 years: {slope * 26:.1f} mm")
print(f"  Statistical significance: p = {p_value:.4f} ({'significant' if p_value < 0.05 else 'not significant'})")

In [ ]:
# Seasonal precipitation analysis
seasonal_precip = monthly.groupby(['year', 'season']).agg({
    'precip_total_mm': 'sum'
}).reset_index()

# Pivot for heatmap
seasonal_precip_pivot = seasonal_precip.pivot(index='year', columns='season', values='precip_total_mm')
seasonal_precip_pivot = seasonal_precip_pivot[['winter', 'spring', 'summer', 'fall']]

# Plot heatmap
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(seasonal_precip_pivot, annot=True, fmt='.0f', cmap='Blues', 
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Precipitation (mm)'})
ax.set_title('Seasonal Precipitation by Year (mm)', fontsize=14)
ax.set_xlabel('Season')
ax.set_ylabel('Year')

plt.tight_layout()
plt.savefig('notebooks/output/seasonal_precip_heatmap.png', dpi=150)
plt.show()

In [ ]:
# Winter precipitation trend (key question: is winter rain decreasing?)
winter_precip = seasonal_precip[seasonal_precip['season'] == 'winter']

fig, ax = plt.subplots(figsize=(10, 5))

ax.bar(winter_precip['year'], winter_precip['precip_total_mm'], 
       color='lightblue', edgecolor='steelblue')

# Trend line
slope, intercept, r_value, p_value, std_err = stats.linregress(
    winter_precip['year'], winter_precip['precip_total_mm']
)
ax.plot(winter_precip['year'], intercept + slope * winter_precip['year'], 
        'r-', linewidth=2, label=f'Trend: {slope:.1f} mm/year')

ax.set_xlabel('Year')
ax.set_ylabel('Winter Precipitation (mm)')
ax.set_title('Winter Precipitation Trend (Dec-Feb)', fontsize=14)
ax.legend()

ax.text(0.02, 0.98, f'p = {p_value:.3f}', 
        transform=ax.transAxes, fontsize=10, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig('notebooks/output/winter_precip_trend.png', dpi=150)
plt.show()

print(f"\nWinter Precipitation Trend:")
print(f"  Change: {slope:.2f} mm/year")
print(f"  Over 26 years: {slope * 26:.1f} mm")
print(f"  Statistical significance: p = {p_value:.4f} ({'significant' if p_value < 0.05 else 'not significant'})")

## 5. Growing Degree Days (GDD) Analysis

Analyze how GDD accumulation is changing over time, which affects crop development.

In [ ]:
# Annual GDD totals
annual_gdd = monthly.groupby('year').agg({
    'gdd_total': 'sum'
}).reset_index()

print("Annual GDD (Base 10°C):")
annual_gdd

In [ ]:
# Plot GDD trend
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(annual_gdd['year'], annual_gdd['gdd_total'], 'o-', 
        color='orange', linewidth=2, markersize=6)

# Trend line
slope, intercept, r_value, p_value, std_err = stats.linregress(
    annual_gdd['year'], annual_gdd['gdd_total']
)
ax.plot(annual_gdd['year'], intercept + slope * annual_gdd['year'], 
        'r--', linewidth=2, label=f'Trend: {slope:.1f} degree-days/year')

ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Annual GDD (degree-days)', fontsize=12)
ax.set_title('Growing Degree Days Trend (Base 10°C)', fontsize=14)
ax.legend()

ax.text(0.02, 0.98, f'R² = {r_value**2:.3f}\np-value = {p_value:.4f}', 
        transform=ax.transAxes, fontsize=10, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig('notebooks/output/gdd_trend.png', dpi=150)
plt.show()

print(f"\nGDD Trend Analysis:")
print(f"  Rate of change: {slope:.2f} degree-days per year")
print(f"  Total change over 26 years: {slope * 26:.1f} degree-days")
print(f"  Statistical significance: p = {p_value:.4f} ({'significant' if p_value < 0.05 else 'not significant'})")

In [ ]:
# Seasonal GDD
seasonal_gdd = monthly.groupby(['year', 'season']).agg({
    'gdd_total': 'sum'
}).reset_index()

# Focus on growing season (spring + summer)
growing_season = seasonal_gdd[seasonal_gdd['season'].isin(['spring', 'summer'])]
growing_by_year = growing_season.groupby('year')['gdd_total'].sum().reset_index()

fig, ax = plt.subplots(figsize=(10, 5))

ax.bar(growing_by_year['year'], growing_by_year['gdd_total'], 
       color='orange', alpha=0.7)

# Trend line
slope, intercept, r_value, p_value, std_err = stats.linregress(
    growing_by_year['year'], growing_by_year['gdd_total']
)
ax.plot(growing_by_year['year'], intercept + slope * growing_by_year['year'], 
        'r-', linewidth=2, label=f'Trend: {slope:.1f} degree-days/year')

ax.set_xlabel('Year')
ax.set_ylabel('GDD (degree-days)')
ax.set_title('Growing Season GDD (Spring + Summer)', fontsize=14)
ax.legend()

plt.tight_layout()
plt.savefig('notebooks/output/growing_season_gdd.png', dpi=150)
plt.show()

## 6. Summary Statistics

In [ ]:
# Compile summary statistics
summary_data = []

# Temperature trend
slope_t, _, r_t, p_t, _ = stats.linregress(annual_temp['year'], annual_temp['temp_mean_c'])
summary_data.append({
    'Metric': 'Annual Temperature',
    'Trend (per year)': f'{slope_t:+.4f}°C',
    '26-year change': f'{slope_t*26:+.2f}°C',
    'R²': f'{r_t**2:.3f}',
    'p-value': f'{p_t:.4f}',
    'Significant': 'Yes' if p_t < 0.05 else 'No'
})

# Precipitation trend
slope_p, _, r_p, p_p, _ = stats.linregress(annual_precip['year'], annual_precip['precip_total_mm'])
summary_data.append({
    'Metric': 'Annual Precipitation',
    'Trend (per year)': f'{slope_p:+.1f} mm',
    '26-year change': f'{slope_p*26:+.1f} mm',
    'R²': f'{r_p**2:.3f}',
    'p-value': f'{p_p:.4f}',
    'Significant': 'Yes' if p_p < 0.05 else 'No'
})

# GDD trend
slope_g, _, r_g, p_g, _ = stats.linregress(annual_gdd['year'], annual_gdd['gdd_total'])
summary_data.append({
    'Metric': 'Annual GDD',
    'Trend (per year)': f'{slope_g:+.1f} days',
    '26-year change': f'{slope_g*26:+.1f} days',
    'R²': f'{r_g**2:.3f}',
    'p-value': f'{p_g:.4f}',
    'Significant': 'Yes' if p_g < 0.05 else 'No'
})

# Winter precipitation trend
slope_w, _, r_w, p_w, _ = stats.linregress(winter_precip['year'], winter_precip['precip_total_mm'])
summary_data.append({
    'Metric': 'Winter Precipitation',
    'Trend (per year)': f'{slope_w:+.1f} mm',
    '26-year change': f'{slope_w*26:+.1f} mm',
    'R²': f'{r_w**2:.3f}',
    'p-value': f'{p_w:.4f}',
    'Significant': 'Yes' if p_w < 0.05 else 'No'
})

summary_df = pd.DataFrame(summary_data)
print("=" * 80)
print("CLIMATE TREND SUMMARY (2000-2025)")
print("=" * 80)
print(summary_df.to_string(index=False))
print("=" * 80)

In [ ]:
# Key Findings
print("\n" + "=" * 80)
print("KEY FINDINGS")
print("=" * 80)

# Temperature
if slope_t > 0 and p_t < 0.05:
    print(f"✓ temperatures are INCREASING significantly ({slope_t*26:.1f}°C over 26 years)")
elif slope_t > 0:
    print(f"• Temperatures appear to be increasing but not significantly")
else:
    print(f"• Temperatures are stable or decreasing")

# Summer temperature
summer_data = seasonal_temp[seasonal_temp['season'] == 'summer']
slope_s, _, _, p_s, _ = stats.linregress(summer_data['year'], summer_data['temp_mean_c'])
if slope_s > 0 and p_s < 0.05:
    print(f"✓ SUMMERS ARE GETTING HOTTER ({slope_s*26:.1f}°C warmer over 26 years)")
elif slope_s > 0:
    print(f"• Summers may be getting hotter but not significantly")

# Winter precipitation
if slope_w < 0 and p_w < 0.05:
    print(f"✓ WINTER RAIN IS DECREASING significantly ({abs(slope_w*26):.0f}mm less over 26 years)")
elif slope_w < 0:
    print(f"• Winter rain appears to be decreasing but not significantly")

# GDD
if slope_g > 0 and p_g < 0.05:
    print(f"✓ GDD IS INCREASING significantly ({slope_g*26:.0f} more degree-days over 26 years)")
elif slope_g > 0:
    print(f"• GDD appears to be increasing but not significantly")

print("=" * 80)